# Published findings

TSY floor: delivery cycle (effort / wall-clock / impact) → opportunity quadrant → dE/dK/dI register → promoted detail → starter tabs.
Restart & Run All. Numbers from `streamlit/data/*.yaml` unless a tab's views exist in the contract.
Do not re-classify Action Register tags. Opportunity names, not agents. dK is Team Upskill Engine **input** only.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "streamlit").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "streamlit"))

from story_data import load_yaml, scored_stages
from story_visuals import cycle_figure, quadrant_figure, tag_incidence_figure
from tab_athena import (
    DEFAULT_WINDOW_DAYS,
    change_metrics,
    commits_metrics,
    daily_bar,
    empty_cohort,
    load_metrics,
    load_tab_frame,
    pipeline_metrics,
    skip_reason,
    work_item_metrics,
)
import plotly.express as px

VIEWS = load_yaml("contract_views.yaml")["views"]
print("contract views", VIEWS)

## 1. Delivery cycle

In [ ]:
cards = scored_stages()
cycle_figure(cards).show()
for card in cards:
    print(card["name"], "effort", card["effort"], "wall", card["wall_clock"], "impact", card["impact"], "ToC1" if card["toc_rank_1"] else "")

## 2. Opportunity quadrant

In [ ]:
quadrant_figure().show()
for row in load_yaml("ai_opportunities.yaml")["opportunities"]:
    print(row["id"], row["name"], row["ai_can_help"], row["effort"], row["expected_gain"])

## 3. dE / dK / dI analysis

Multi-tag allowed. "Knows the right move, environment punishes it" stays **dE**, not dK.
Routes: dE+eliminate → drop; dE+automate/delegate → AI SDLC candidate; other dE → process/tooling; dK → Team Upskill Engine input; dI → staffing.
Do not author TUE modules.

In [ ]:
import pandas as pd
from story_data import route_for_tag

rows = load_yaml("action_register.yaml")["rows"]
tag_incidence_figure(rows).show()
flat = []
for row in rows:
    for tag in row["cause_tags"]:
        flat.append({
            "rank": row["rank"],
            "finding": row["finding"],
            "decision": row["decision"],
            "tag": tag,
            "route": row["routes"].get(tag) or route_for_tag(tag, row["decision"]),
            "n": row["n"],
            "role": row["role"],
            "clock": row["clock"],
            "unit": row["unit"],
        })
pd.DataFrame(flat)

## 4. Promoted stage detail

In [ ]:
promoted = load_yaml("promoted_eda.yaml").get("rows") or []
if not promoted:
    print("No extra EDA promoted. Opening two visuals stand.")
else:
    display(pd.DataFrame(promoted))

## 5. Tab: Commits

In [ ]:
reason = skip_reason("commits", VIEWS)
if reason:
    print(reason)
else:
    frame, qid = load_tab_frame("commits", DEFAULT_WINDOW_DAYS)
    metrics = commits_metrics(frame, DEFAULT_WINDOW_DAYS)
    print("QueryExecutionId", qid, "n", metrics["n"], "active_authors", metrics["active_authors"])
    if metrics["n"] == 0:
        print("Empty cohort:", empty_cohort("commits", DEFAULT_WINDOW_DAYS))
    else:
        daily_bar(metrics["daily"], "day", "commits", "Commits / day").show()
        px.bar(
            metrics["author_share"].head(20),
            x="author",
            y="commits",
            title="Author commit count (role=author, n in window)",
        ).show()
        px.bar(metrics["type_mix"], x="type", y="commits", title="Conventional-commit mix").show()
        display(metrics["author_share"].head(20))

## 6. Tab: Work items

In [ ]:
reason = skip_reason("work_items", VIEWS)
if reason:
    print(reason)
else:
    frame, qid = load_tab_frame("work_items", DEFAULT_WINDOW_DAYS)
    metrics = work_item_metrics(frame, DEFAULT_WINDOW_DAYS)
    print("QueryExecutionId", qid, "n", metrics["n"])
    print(metrics["clocks_reason"])
    if metrics["n"] == 0:
        print("Empty cohort:", empty_cohort("work_items", DEFAULT_WINDOW_DAYS))
    else:
        daily_bar(metrics["closed_daily"], "day", "closed", "Closed issues / day").show()
        daily_bar(metrics["touched_daily"], "day", "touched", "Touched issues / day").show()

## 7. Tab: Changes

In [ ]:
reason = skip_reason("changes", VIEWS)
if reason:
    print(reason)
else:
    frame, qid = load_tab_frame("changes", DEFAULT_WINDOW_DAYS)
    metrics = change_metrics(frame, DEFAULT_WINDOW_DAYS)
    print(
        "QueryExecutionId",
        qid,
        "opened_n",
        metrics["n"],
        "merged_n",
        metrics["n_merged"],
        "median_hours",
        metrics["median_hours"],
        "unit=hours created→merged (not issue business days)",
    )
    if metrics["n"] == 0:
        print("Empty cohort:", empty_cohort("changes", DEFAULT_WINDOW_DAYS))
    else:
        daily_bar(metrics["opened_daily"], "day", "opened", "PRs opened / day").show()
        daily_bar(metrics["merged_daily"], "day", "merged", "PRs merged / day").show()
        px.bar(
            metrics["integrators"].head(20),
            x="integrator",
            y="n",
            title="Integrators (merge actor, not assigned reviewer)",
        ).show()

## 8. Tab: Pipelines

In [ ]:
reason = skip_reason("pipelines", VIEWS)
if reason:
    print(reason)
else:
    frame, qid = load_tab_frame("pipelines", DEFAULT_WINDOW_DAYS)
    metrics = pipeline_metrics(frame, DEFAULT_WINDOW_DAYS)
    print(
        "QueryExecutionId",
        qid,
        "n_terminal",
        metrics["n"],
        "failure_rate",
        metrics["failure_rate"],
        "median_duration_s",
        metrics["median_duration_s"],
    )
    print("Top failing ranks workflow name — job-level rows are not in this lake.")
    if metrics["n"] == 0:
        print("Empty cohort:", empty_cohort("pipelines", DEFAULT_WINDOW_DAYS))
    else:
        px.bar(
            metrics["status_daily"],
            x="day",
            y="runs",
            color="conclusion",
            title="Workflow status by day (cancelled excluded from failure rate)",
        ).show()
        px.bar(metrics["top_failing"], x="failures", y="workflow", orientation="h", title="Top failing workflows").show()

## 9. Tab: Load

In [ ]:
reason = skip_reason("load", VIEWS)
if reason:
    print(reason)
else:
    frame, qid = load_tab_frame("load", DEFAULT_WINDOW_DAYS)
    metrics = load_metrics(frame, DEFAULT_WINDOW_DAYS)
    print(
        "QueryExecutionId",
        qid,
        "n",
        metrics["n"],
        "weekend_share",
        metrics["weekend_share"],
        "long_day_share",
        metrics["long_day_share"],
    )
    print("RISK SIGNAL, not a performance score.", metrics.get("unit"))
    if metrics["n"] == 0:
        print("Empty cohort:", empty_cohort("load", DEFAULT_WINDOW_DAYS))
    else:
        display(metrics["authors_under_load"].head(20))

## 10. Tab: Defects

In [ ]:
reason = skip_reason("defects", VIEWS)
if reason:
    print(reason)
else:
    raise RuntimeError("Defect types are scoped — implement time-to-target from the contract, do not guess")

## 11. Tab: Sprints

In [ ]:
reason = skip_reason("sprints", VIEWS)
if reason:
    print(reason)
else:
    raise RuntimeError("Sprint snapshots are scoped — implement committed vs delivered from those rows")